# Rolling Surrogate Regime Analysis

This notebook implements a **rolling surrogate modeling workflow** for analyzing complex, non-stationary systems whose behavior evolves over time.

Rather than fitting a single global model and treating deviations as noise, this approach explicitly treats **changes in model fit, structure, and explanatory drivers as signals**. The goal is not to maximize predictive performance, but to use lightweight, local models as **diagnostic instruments** for detecting and characterizing regime shifts in a dynamic process.

The workflow is designed to answer three questions:

1. *Is the system currently behaving similarly to its recent past, or has it entered a new regime?*  
2. *When the system changes, which signals or features change in explanatory relevance?*  
3. *Are observed changes likely driven by internal system dynamics, external inputs, or structural reconfiguration?*  

---

## Conceptual Structure

The workflow consists of four conceptual stages:

### 1. Time-Resolved Feature Construction
Raw time series from individual runs are converted into a structured, run-level feature table by:

- Aligning each run in time relative to a defined start event,
- Segmenting each run into a small number of user-defined temporal windows,
- Computing compact summary statistics and dynamic descriptors within each window (e.g., central tendency, trend, variability, autocorrelation, frequency content),
- Optionally computing cross-signal coupling features (lag-aware correlations) to capture responsiveness and coordination between signals.

This produces a wide, per-run feature matrix in which each row represents a run and each column represents a time-localized or cross-signal descriptor.

---

### 2. Rolling Local Surrogate Modeling
A simple, interpretable model (a Random Forest regressor) is trained repeatedly on a **sliding window of recent runs**.

Each model is treated not as a final predictor, but as a *local surrogate* that approximates the system’s current input–output relationship.

For each window the workflow records:

- Out-of-bag performance (as a proxy for local predictability),
- Residual behavior,
- Feature importance rankings.

These quantities are tracked over time and interpreted as **structural diagnostics** rather than optimization metrics.

---

### 3. Regime Detection via Structural Drift
Regime changes are inferred when one or more of the following occurs:

- Predictive performance drops relative to recent history,
- Residuals become systematically biased or unstable,
- The ranking or identity of dominant explanatory features changes.

Rather than defining regimes a priori, regimes emerge from **changes in explanatory structure**.

This reframes regime detection as a question of *model stability* rather than *output deviation alone*.

---

### 4. Attribution and Interpretation
When instability is detected, the workflow generates focused diagnostic outputs:

- The features whose importance changed most strongly,
- Whether those changes are associated with internal system signals, external inputs, or coupling structure,
- Whether removing certain feature groups stabilizes or destabilizes the model (ablation analysis).

This supports interpretive questions like:
- “What kind of change occurred?”
- “Is the system internally reorganizing, externally driven, or structurally constrained?”
- “Is this change transient, persistent, or escalating?”

---

## Design Principles

- **Locality over globality:** The system is modeled only in its recent context, not forced into a single stationary description.  
- **Explainability over optimization:** Models are used as measurement devices for structure, not as production predictors.  
- **Change as signal:** Shifts in model behavior are treated as primary observables, not as errors to be minimized.  
- **Minimal assumptions:** The workflow does not assume specific causal structure, regime count, or regime form.  

---

## Summary

This rolling surrogate regime analysis framework treats a complex process as a sequence of locally coherent behaviors rather than a single stable system. By tracking how small, interpretable models evolve over time, it provides a principled way to detect regime shifts, attribute their drivers, and reason about structural change — even in systems where mechanistic models are incomplete or unavailable.

It is most useful when:
- the system is adaptive, evolving, or partially controlled,
- causal structure is only partially known,
- and the primary analytical goal is **situational awareness and interpretability**, not pure prediction.

In [ ]:
# Dependency verification for the feature pipeline.
import sys, subprocess, importlib.util, warnings

REQUIRED_PACKAGES = {
    "numpy": "numpy>=1.24",
    "pandas": "pandas>=2.0",
    "matplotlib": "matplotlib>=3.7",
    "scipy": "scipy>=1.11",
    "statsmodels": "statsmodels>=0.14",
    "sklearn": "scikit-learn>=1.3",
    "tqdm": "tqdm>=4.66",
    "ipywidgets": "ipywidgets>=8.0",
    "antropy": "antropy>=0.1.6",
}

for import_name, package_spec in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package_spec}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package_spec])

try:
    from notebook.nbextensions import enable_nbextension
    enable_nbextension("notebook", "widgets/notebook/js/extension")
except Exception:
    pass

import os, re, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from scipy.signal import periodogram
from scipy import signal, stats

import statsmodels.api as sm
from statsmodels.tsa.stattools import acf
from statsmodels.nonparametric.smoothers_lowess import lowess

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore", category=FutureWarning)
print("Dependency check complete.")

In [ ]:
# Raw time-series files -> cleaned per-run dataframes.

# ----------------------------- CONFIG ---------------------------------------
RAW_TIMESERIES_DIR = Path("pipeline_inputs/time_series")
RAW_TIMESERIES_GLOB = "*.csv"

RUN_METADATA_CSV = Path("pipeline_inputs/run_metadata.csv")

PROCESSED_TIMESERIES_DIR = Path("pipeline_outputs/processed_time_series")
PLOT_OUTPUT_DIR = Path("pipeline_outputs/plots")
FEATURE_OUTPUT_DIR = Path("pipeline_outputs/features")

FEATURE_TABLE_CSV = "feature_table.csv"
FULL_FEATURE_TABLE_CSV = "feature_table_full.csv"
TRAIN_FEATURES_CSV = "training_features.csv"
TRAIN_TARGET_CSV = "training_target.csv"

PRIMARY_TARGET_COL = "target_metric_primary"
SOURCE_TARGET_COL = "source_target_metric"

SMOOTH_WIN = 30

SIGNAL_PATTERNS = [
    "ai/{unit}_feature_7/eu",
    "{unit}_feature_1_block/eu",
    "{unit}_feature_1_block/setpoint",
    "{unit}_feature_3/eu",
    "ai/{unit}_feature_4_1/eu",
    "ai/{unit}_feature_8/eu",
    "ai/{unit}_feature_6/eu",
    "{unit}_feature_5",
]
# ---------------------------------------------------------------------------

UNIT_RE = re.compile(r"UNIT\d+|GR\d\d", re.I)
RUN_RE = re.compile(r"(?:^|[_\-\s])(F?RUN[_\-]?0*\d+[A-Z]?|F?BH0*\d+[A-Z]?)(?:$|[_\-\s])", re.I)

def normalize_run_id(value: object) -> str:
    """Return a stable anonymized run identifier."""
    text = str(value).strip().upper()
    text = re.sub(r"^F", "", text)

    run_match = re.search(r"RUN[_\-]?0*(\d+)([A-Z]?)", text)
    if run_match:
        number, suffix = run_match.groups()
        return f"RUN_{int(number):03d}{suffix}"

    generic_match = re.search(r"BH0*(\d+)([A-Z]?)", text)
    if generic_match:
        number, suffix = generic_match.groups()
        return f"RUN_{int(number):03d}{suffix}"

    return re.sub(r"\s+", "_", text)

def extract_run_id_from_name(name: str) -> str | None:
    """Extract a normalized run identifier from a filename or free-text label."""
    padded = f"_{name}_"
    match = RUN_RE.search(padded)
    return normalize_run_id(match.group(1)) if match else None

def detect_unit(cols: list[str]) -> str:
    """Return the first anonymized unit token found in the column headers."""
    for col in cols:
        match = UNIT_RE.search(col)
        if match:
            return match.group().upper()
    raise ValueError("No unit identifier found in the header")

def target_columns(unit: str) -> list[str]:
    """Expand configured signal patterns for a unit id."""
    return [f"{unit}/{pattern.format(unit=unit)}" for pattern in SIGNAL_PATTERNS]

def load_run_windows(metadata_csv: Path) -> dict[str, tuple[pd.Timestamp, pd.Timestamp]]:
    """Build run_id -> analysis time-window mapping from the metadata file."""
    metadata = pd.read_csv(metadata_csv)
    required = ["source_run_id", "source_run_start_date", "source_run_end_date"]
    missing = [col for col in required if col not in metadata.columns]
    if missing:
        raise KeyError(f"Run metadata file is missing required columns: {missing}")

    metadata["_start_date"] = pd.to_datetime(metadata["source_run_start_date"], errors="coerce").dt.normalize()
    metadata["_end_date"] = pd.to_datetime(metadata["source_run_end_date"], errors="coerce").dt.normalize()

    start_ts = metadata["_start_date"] + pd.to_timedelta(17, "h")
    end_ts = metadata["_end_date"] + pd.to_timedelta(6, "h")

    return {
        normalize_run_id(run_id): (start, end)
        for run_id, start, end in zip(metadata["source_run_id"], start_ts, end_ts)
        if pd.notna(start) and pd.notna(end)
    }

RUN_TIME_WINDOWS = load_run_windows(RUN_METADATA_CSV)

def process_csv(path: Path, plot: bool = False) -> dict:
    """Return cleaned run data, smoothed signal names, and unit id."""
    df = pd.read_csv(path, low_memory=False)
    df["t_stamp"] = pd.to_datetime(df["t_stamp"], errors="coerce")
    df = df.dropna(subset=["t_stamp"]).sort_values("t_stamp")

    run_id = normalize_run_id(path.stem)
    if run_id in RUN_TIME_WINDOWS:
        start_ts, end_ts = RUN_TIME_WINDOWS[run_id]
        cropped = df[(df["t_stamp"] >= start_ts) & (df["t_stamp"] <= end_ts)].copy()
        if cropped.empty:
            print(f"{path.name}: no rows inside configured run window; keeping full file.")
        else:
            df = cropped
    else:
        print(f"{path.name}: no matching run metadata; no time crop applied.")

    stale_setpoint_cols = [col for col in df.columns if col.endswith("feature_1_block/setpoint_sm")]
    if stale_setpoint_cols:
        df.drop(columns=stale_setpoint_cols, inplace=True, errors="ignore")

    unit = detect_unit(df.columns)
    cols = [col for col in target_columns(unit) if col in df.columns]
    if not cols:
        raise ValueError(f"{path.name}: no configured signal columns found")

    df = df[["t_stamp"] + cols].copy()

    for col in cols:
        df.loc[df[col] == 0, col] = np.nan
        if "feature_1_block/eu" in col:
            df[col] = df[col].clip(upper=100)

    paired_a = f"{unit}/ai/{unit}_feature_4_1/eu"
    paired_b = f"{unit}/ai/{unit}_feature_4_2/eu"
    paired_mean = f"{unit}/ai/{unit}_feature_4_mean/eu"

    available_pair_cols = [col for col in [paired_a, paired_b] if col in df.columns]
    if available_pair_cols:
        df[paired_mean] = df[available_pair_cols].mean(axis=1, skipna=True)
        cols = [col for col in cols if col not in (paired_a, paired_b)] + [paired_mean]
        df = df[["t_stamp"] + cols].copy()

    for col in cols:
        df[col + "_sm"] = df[col].rolling(SMOOTH_WIN, center=True, min_periods=1).mean()

    if plot:
        plot_df(df, [col + "_sm" for col in cols], run_id, unit)

    return {
        "data": df,
        "smoothed_columns": [col + "_sm" for col in cols],
        "unit_id": unit,
    }

def plot_df(df, sm_cols, run_name, unit):
    """Plot cleaned signal traces for a single run."""
    plt.figure(figsize=(14, 2.0 * len(sm_cols)))
    for i, col in enumerate(sm_cols, 1):
        plt.subplot(len(sm_cols), 1, i)
        plt.plot(df["t_stamp"], df[col], lw=.8)
        plt.title(f"{run_name} - {col.replace('_sm','')} ({unit})")
        plt.grid(alpha=.3)
    plt.tight_layout()
    plt.show()

def batch_process(data_dir=RAW_TIMESERIES_DIR, glob_pat=RAW_TIMESERIES_GLOB,
                  save_csv=True, show_plots=False):
    """Process all configured raw time-series files."""
    out = {}
    PROCESSED_TIMESERIES_DIR.mkdir(parents=True, exist_ok=True)
    for file_path in tqdm(sorted(Path(data_dir).glob(glob_pat)), desc="Files"):
        try:
            run_id = normalize_run_id(file_path.stem)
            res = process_csv(file_path, plot=show_plots)
            out[run_id] = res
            if save_csv:
                out_path = PROCESSED_TIMESERIES_DIR / f"{run_id}_{res['unit_id']}_processed.csv"
                res["data"].to_csv(out_path, index=False)
        except Exception as exc:
            print(f"{file_path.name}: {exc}")
    return out

processed_runs = batch_process(show_plots=False)
print(f"Processed {len(processed_runs)} runs into dict 'processed_runs'.")

In [ ]:
# Merge optional supplemental time-series metrics.

SUPPLEMENTAL_TIMESERIES_DIR = Path("pipeline_inputs/supplemental_time_series")
SUPPLEMENTAL_TIMESERIES_GLOB = "*.csv"

def index_supplemental_files(supplemental_dir=SUPPLEMENTAL_TIMESERIES_DIR,
                             glob_pat=SUPPLEMENTAL_TIMESERIES_GLOB):
    """Build run_id -> list[(unit_id, path)] from supplemental filenames."""
    index = {}
    for file_path in Path(supplemental_dir).glob(glob_pat):
        run_id = extract_run_id_from_name(file_path.stem)
        if run_id is None:
            print(f"{file_path.name}: no run id found in filename; skipped.")
            continue
        unit_match = UNIT_RE.search(file_path.stem)
        unit_id = unit_match.group().upper() if unit_match else "UNIT_UNKNOWN"
        index.setdefault(run_id, []).append((unit_id, file_path))
    return index

supplemental_index = index_supplemental_files()
print(f"Indexed supplemental files for {len(supplemental_index)} runs.")

def load_supplemental_series(path: Path) -> pd.DataFrame:
    """Load supplemental metrics and return standardized columns."""
    df = pd.read_csv(path, low_memory=False)
    df["t_stamp"] = pd.to_datetime(df["t_stamp"], errors="coerce")
    df = df.dropna(subset=["t_stamp"]).sort_values("t_stamp")

    metric_patterns = [
        "feature_10_ext_metric",
        "feature_10_alt_metric",
    ]

    metric_candidates = []
    for pattern in metric_patterns:
        metric_candidates.extend([col for col in df.columns if pattern.lower() in col.lower()])

    if not metric_candidates:
        raise ValueError(f"{path.name}: no column containing any of {metric_patterns}")

    if len(metric_candidates) > 1:
        print(f"{path.name}: multiple supplemental metric columns found; using first.")

    metric_col = metric_candidates[0]
    out = df[["t_stamp", metric_col]].rename(columns={metric_col: "feature_10_ext_metric"})

    secondary_candidates = [col for col in df.columns if "feature_9" in col.lower()]
    if secondary_candidates:
        if len(secondary_candidates) > 1:
            print(f"{path.name}: multiple feature_9 columns found; using first.")
        out["feature_9"] = df[secondary_candidates[0]]

    return out

merged_count = 0

for run_id, res in processed_runs.items():
    if run_id not in supplemental_index:
        print(f"No supplemental file found for run {run_id}")
        continue

    (supp_unit, supp_path), *others = supplemental_index[run_id]
    if others:
        print(f"Multiple supplemental files for run {run_id}; using {supp_path.name}")

    try:
        supp_df = load_supplemental_series(supp_path)
    except Exception as exc:
        print(f"{supp_path.name}: {exc}")
        continue

    df = res["data"].copy()
    df["t_stamp"] = pd.to_datetime(df["t_stamp"], errors="coerce")
    df = df.dropna(subset=["t_stamp"]).sort_values("t_stamp")

    merged = pd.merge_asof(
        df.sort_values("t_stamp"),
        supp_df.sort_values("t_stamp"),
        on="t_stamp",
        direction="nearest",
        tolerance=pd.Timedelta("30s"),
    )

    for metric_name in ["feature_10_ext_metric", "feature_9"]:
        if metric_name not in merged.columns:
            continue
        sm_col = metric_name + "_sm"
        if sm_col not in merged.columns:
            merged[sm_col] = merged[metric_name].rolling(SMOOTH_WIN, center=True, min_periods=1).mean()
        if sm_col not in res["smoothed_columns"]:
            res["smoothed_columns"].append(sm_col)

    res["data"] = merged
    processed_runs[run_id] = res

    out_path = PROCESSED_TIMESERIES_DIR / f"{run_id}_{res['unit_id']}_processed.csv"
    merged.to_csv(out_path, index=False)

    merged_count += 1
    print(f"Merged supplemental data into run {run_id} -> {out_path.name}")

print(f"Completed supplemental merge for {merged_count} runs.")

In [ ]:
# Example: visualize one processed run.
process_csv(RAW_TIMESERIES_DIR / "RUN_049.csv", plot=True)

In [ ]:
# ╔═══════════════════════════════════════════════════════════╗
# ║  Timestamp → time-from-start  +  ROI masks (vectorised)   ║
# ╚═══════════════════════════════════════════════════════════╝

# ---- configuration ----------------------------------------------------------
# Define the three ROI time windows in *hours from start*.
# Edit these values/labels to whatever time frames make sense for your data.
ROI_CUTOFF_1 = 100.0   # end of WINDOW_1
ROI_CUTOFF_2 = 200.0   # end of WINDOW_2 (WINDOW_3 starts here)

ROI_LABELS = ("WINDOW_1", "WINDOW_2", "WINDOW_3")

TIME_ROIS = {
    ROI_LABELS[0]: (0, ROI_CUTOFF_1),
    ROI_LABELS[1]: (ROI_CUTOFF_1, ROI_CUTOFF_2),
    ROI_LABELS[2]: (ROI_CUTOFF_2, np.inf),
}
ROI_NAMES = list(TIME_ROIS)        # keep order for plotting

feature_data = {
    k: {
        "time_series": v["data"].copy(),
        "smoothed_columns": v["smoothed_columns"]
    } for k, v in processed_runs.items()
}

# ---- worker ---------------------------------------------------------------
def add_time_from_start(df):
    """Return df with new 'time_from_start' col in hours (float)."""
    t0 = df["t_stamp"].iloc[0]
    return df.assign(time_from_start=(df["t_stamp"] - t0).dt.total_seconds() / 3600)

def roi_masks(hours):
    """Return a {roi: bool-array} dict for a 1-D hours array."""
    return {roi: (hours >= lo) & (hours < hi) for roi, (lo, hi) in TIME_ROIS.items()}

# ---- main loop over runs ---------------------------------------------------
if "feature_data" not in globals():
    raise RuntimeError("Run the earlier preprocessing cell first (builds feature_data)")

processed_runs        = {}
enhanced_feature_data = {}

print(f"⏱  Creating time-ROI masks for {len(feature_data)} runs …\n")
for run, meta in tqdm(feature_data.items(), desc="Runs"):
    df = meta["time_series"].reset_index().sort_values("t_stamp").reset_index(drop=True)
    df = add_time_from_start(df)

    masks = roi_masks(df["time_from_start"].values)

    processed_runs[run] = {
        "data"              : df,
        "roi_masks"         : masks,
        "smoothed_columns"  : meta["smoothed_columns"],
        "start_time"        : df["t_stamp"].iloc[0],
        "total_duration_h"  : df["time_from_start"].iloc[-1],
        "total_points"      : len(df),
    }

    enhanced_feature_data[run] = {
        "run_name"          : run,
        "data"              : df,
        "roi_masks"         : masks,
        "smoothed_columns"  : meta["smoothed_columns"],
        "metadata"          : {
            "start_time"        : df["t_stamp"].iloc[0],
            "total_duration_h"  : df["time_from_start"].iloc[-1],
            "total_points"      : len(df),
        },
    }

print(f"\n✅  Finished: {len(enhanced_feature_data)} runs processed.")

# ╔═══════════════════════════════════════════════════════════╗
# ║  Quick ROI-boundary visual (optional)                     ║
# ╚═══════════════════════════════════════════════════════════╝
def plot_roi(sample_run="RUN_020", sample_sensor_key="feature_4_mean"):
    if sample_run not in enhanced_feature_data:
        print(f"{sample_run} not found."); return
    run_dict = enhanced_feature_data[sample_run]
    df       = run_dict["data"]

    # pick first smoothed column containing the key
    try:
        sensor_col = next(c for c in run_dict["smoothed_columns"] if sample_sensor_key in c)
    except StopIteration:
        print(f"Sensor key '{sample_sensor_key}' not in run."); return

    plt.figure(figsize=(13,5))
    plt.plot(df["time_from_start"], df[sensor_col], lw=.8, label=sensor_col)

    for i, roi in enumerate(ROI_NAMES):
        plt.axvline(TIME_ROIS[roi][0], color=f"C{i}", ls="--", alpha=.7, label=f"{roi} start")

    plt.xlabel("Hours from start"); plt.ylabel("Sensor value")
    plt.title(f"{sample_run} – ROI boundaries on '{sensor_col}'")
    plt.legend(ncol=3); plt.grid(alpha=.2); plt.tight_layout(); plt.show()

# Example usage:
plot_roi("RUN_049", "feature_5")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Time-ROI feature extraction → WIDE per-run format           ║
# ╚══════════════════════════════════════════════════════════════╝

# ─────────────────── safety guard ──────────────────────────────
if "enhanced_feature_data" not in globals():
    raise RuntimeError("`enhanced_feature_data` missing – run previous cells first")

# ─────────────────── helpers ───────────────────────────────────
def _features_slim(vals, t):
    """Compact, non-redundant feature set for a 1-D series in an ROI."""
    v = vals.astype(float); t = t.astype(float)
    lr    = LinearRegression().fit(t[:, None], v)
    slope = float(lr.coef_[0])
    resid = v - lr.predict(t[:, None])

    median    = float(np.nanmedian(v))
    resid_std = float(np.nanstd(resid, ddof=0))

    try:
        lag1_ac = float(acf(resid, nlags=2, fft=False)[1])
    except Exception:
        lag1_ac = np.nan

    if resid.size >= 16 and np.nanvar(resid) > 0:
        freqs, pwr = periodogram(resid)
        dom_freq = float(freqs[np.argmax(pwr)]) if pwr.size else np.nan
    else:
        dom_freq = np.nan

    return dict(median=median,
                slope=slope,
                resid_std=resid_std,
                lag1_autocorr=lag1_ac,
                dominant_freq=dom_freq)

def _is_feature_3_sensor(sm_name: str) -> bool:
    # We only special-case the smoothed setpoint column
    return "feature_1_block/setpoint_sm" in sm_name

def _raw_from_sm(sm_name: str) -> str:
    return sm_name[:-3] if sm_name.endswith("_sm") else sm_name

def _friendly_sensor_key(colname: str) -> str:
    """
    Make a short, unit-agnostic sensor key from a full column path.
    """
    parts = colname.split("/")
    if len(parts) >= 2:
        token = parts[-2] 
        if token.startswith("GR") and token == parts[0]:
            # fallback: use final part
            token = parts[-1]
    else:
        token = parts[-1]
    token = token.replace("_sm", "")
    return re.sub(r"^GR\d\d_", "", token)

def _get_run_setpoint(df: pd.DataFrame) -> float:
    """Return a single configured setpoint value for the run."""
    # Prefer raw setpoint (no _sm); fallback to smoothed if needed
    raw_cols = [c for c in df.columns if c.endswith("feature_1_block/setpoint")]
    sm_cols  = [c for c in df.columns if c.endswith("feature_1_block/setpoint_sm")]
    for cols in (raw_cols, sm_cols):
        for c in cols:
            series = df[c].dropna()
            if not series.empty:
                return float(series.iloc[0])
    return np.nan

# Feature names used for non-setpoint sensors (for stable column ordering)
FEATURE_ORDER = list(_features_slim(np.arange(16), np.arange(16)).keys())

# ─────────────────── extraction → wide rows per run ────────────
wide_rows = []
print(f"\n🔎 Extracting features (wide per-run) for {len(enhanced_feature_data)} runs …")
for run, r in tqdm(enhanced_feature_data.items(), desc="Runs"):
    df, masks, sensors = r["data"], r["roi_masks"], r["smoothed_columns"]

    # Initialize one wide row per run
    row = {"run_id": run, "feature_1_setpoint": _get_run_setpoint(df)}

    # Iterate ROIs in the original order
    for roi, m in masks.items():
        sub = df.loc[m]
        if sub.empty:
            continue
        times = sub["time_from_start"].values

        for sensor in sensors:
            # Special handling: skip regular features for setpoint (we already captured per-run value)
            if _is_feature_3_sensor(sensor):
                continue
            if sensor not in sub.columns:
                continue

            vals = sub[sensor].values
            valid = np.isfinite(vals)
            sensor_key = _friendly_sensor_key(sensor)

            # ✅ SPECIAL CASE: feature_8/eu → ONLY mean per ROI (no full featurization)
            if "feature_8/eu" in sensor:
                mean_val = float(np.nanmean(vals[valid])) if valid.any() else np.nan
                row[f"{sensor_key}_mean_{roi}"] = mean_val
                continue

            # Default behavior: full feature set for all other sensors
            if valid.sum() < 5 or np.nanvar(vals) == 0:
                feats = {k: np.nan for k in FEATURE_ORDER}
            else:
                feats = _features_slim(vals[valid], times[valid])

            # Add to the wide row: <sensor>_<feature>_<ROI>
            for feat_name in FEATURE_ORDER:
                col = f"{sensor_key}_{feat_name}_{roi}"
                row[col] = feats[feat_name]

    wide_rows.append(row)

# ─────────────────── assemble + save ───────────────────────────
feat_df = pd.DataFrame(wide_rows)

# Ensure 'Run' then 'feature_1_setpoint' appear first, followed by others in a reproducible order
front_cols = ["run_id", "feature_1_setpoint"]
other_cols = sorted([c for c in feat_df.columns if c not in front_cols])
feat_df = feat_df[front_cols + other_cols]

out_dir = Path("pipeline_outputs/features")
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / "feature_table.csv"
feat_df.to_csv(out_csv, index=False)

print(f"\n✅  Done: {len(feat_df)} runs")
print(f"📄  CSV saved → {out_csv.resolve()}")
feat_df.head()

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CCF on cleaned data (per ROI) → append CAI to wide features ║
# ╚══════════════════════════════════════════════════════════════╝

# ---- Config ---------------------------------------------------------------
MAX_LAG_MIN = 30         # ± minutes for CCF lag window
TAU_MIN     = 15.0       # CAI time constant (minutes)
SMOOTH_WIN  = globals().get("SMOOTH_WIN", 30)  # fallback if not defined

# ---- Safety ---------------------------------------------------------------
for _name in ("enhanced_feature_data", "feat_df"):
    if _name not in globals():
        raise RuntimeError(f"Required object `{_name}` not found. Run previous cells first.")

# Respect your ROI order
ROI_ORDER = list(TIME_ROIS.keys()) if "TIME_ROIS" in globals() else list(next(iter(enhanced_feature_data.values()))["roi_masks"].keys())

# ---- Helpers --------------------------------------------------------------
def _zscore(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    mu, sd = np.nanmean(x), np.nanstd(x, ddof=1)
    return (x - mu) / sd if np.isfinite(sd) and sd > 0 else np.full_like(x, np.nan)

def _ccf_pearson(x: np.ndarray, y: np.ndarray, lag_max: int) -> dict[int, float]:
    out = {}; n = len(x)
    for k in range(-lag_max, lag_max + 1):
        if k >= 0: xs, ys = x[k:], y[: n - k]
        else:      xs, ys = x[: n + k], y[-k:]
        out[k] = np.corrcoef(xs, ys)[0, 1] if len(xs) >= 2 else np.nan
    return out

def _median_dt_minutes(ts: pd.Series) -> float:
    if len(ts) < 2: return 1.0
    diffs = np.diff(ts.to_numpy("datetime64[ns]")).astype("timedelta64[s]").astype(float) / 60.0
    v = float(np.nanmedian(diffs)) if np.isfinite(diffs).any() else 1.0
    return v if v > 0 and np.isfinite(v) else 1.0

def _cai(abs_corr: float, lag_minutes: float, tau_min: float = TAU_MIN) -> float:
    return float(abs_corr * np.exp(-abs(lag_minutes) / tau_min))

def _find_col(df_cols, substrings):
    """Return the first column whose name contains ANY of the substrings."""
    for sub in substrings:
        hits = [c for c in df_cols if sub in c]
        if len(hits) == 1:
            return hits[0]
        if len(hits) > 1:  # if multiple, prefer the shortest (usually no duplicate sensors)
            return sorted(hits, key=len)[0]
    return None

def _get_series(df: pd.DataFrame, mask: pd.Series, prefer_sm_sub: str, prefer_raw_sub: str) -> np.ndarray | None:
    """Pick smoothed column by substring; fallback to raw and smooth on the fly."""
    # Try smoothed
    sm_col = _find_col(df.columns, [prefer_sm_sub])
    if sm_col and sm_col in df.columns:
        return pd.to_numeric(df.loc[mask, sm_col], errors="coerce").to_numpy(dtype=float)
    # Try raw
    raw_col = _find_col(df.columns, [prefer_raw_sub])
    if raw_col and raw_col in df.columns:
        ser = pd.to_numeric(df.loc[mask, raw_col], errors="coerce")
        ser = ser.rolling(SMOOTH_WIN, center=True, min_periods=1).mean()  # smooth on the fly
        return ser.to_numpy(dtype=float)
    return None

# ---- Main: build per-run CAI across ROIs ---------------------------------
cai_rows = []
print(f"\n🔗 Computing signal↔reference CCF/CAI for {len(enhanced_feature_data)} runs …")
for run, r in tqdm(enhanced_feature_data.items(), desc="Runs"):
    df   = r["data"]
    masks= r["roi_masks"]

    row = {"run_id": run}

    for roi in ROI_ORDER:
        m = masks.get(roi, None)
        if m is None or df.loc[m].empty:
            row[f"cai_f1_over_f3_{roi}"] = np.nan
            continue

        sub = df.loc[m]

        # Find signal and reference series (prefer *_sm; else raw)
        signal_series = _get_series(
            df, m,
            prefer_sm_sub="feature_1_block/eu_sm",
            prefer_raw_sub="feature_1_block/eu",
        )
        mst = _get_series(
            df, m,
            prefer_sm_sub="feature_3/eu_sm",
            prefer_raw_sub="feature_3/eu",
        )

        if signal_series is None or mst is None:
            row[f"cai_f1_over_f3_{roi}"] = np.nan
            continue

        valid = np.isfinite(signal_series) & np.isfinite(mst)
        if valid.sum() < 5:
            row[f"cai_f1_over_f3_{roi}"] = np.nan
            continue

        signal_z, mst_z = _zscore(signal_series[valid]), _zscore(mst[valid])
        ts_roi = sub.loc[valid, "t_stamp"]

        dt_min = _median_dt_minutes(ts_roi)
        lag_max_samples = max(1, int(round(MAX_LAG_MIN / dt_min)))

        corrs = _ccf_pearson(signal_z, mst_z, lag_max=lag_max_samples)
        valid_corrs = {k: v for k, v in corrs.items() if np.isfinite(v)}
        if not valid_corrs:
            row[f"cai_f1_over_f3_{roi}"] = np.nan
            continue

        best_k   = max(valid_corrs, key=lambda k: abs(valid_corrs[k]))
        best_r   = float(valid_corrs[best_k])
        best_min = float(best_k * dt_min)

        row[f"cai_f1_over_f3_{roi}"] = _cai(abs_corr=abs(best_r), lag_minutes=best_min)

    cai_rows.append(row)

cai_df = pd.DataFrame(cai_rows)

# ---- Merge into your wide features and save -------------------------------
feat_df = feat_df.merge(cai_df, on="run_id", how="left")

out_dir = Path("pipeline_outputs/features")
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / "feature_table.csv"
feat_df.to_csv(out_csv, index=False)

print(f"\n✅  Appended CAI columns: {[c for c in cai_df.columns if c!='Run']}")
print(f"📄  Updated CSV saved → {out_csv.resolve()}")
feat_df.head()

In [ ]:
# Join run-level metadata into the wide feature table.

for required_name in ("feat_df", "enhanced_feature_data"):
    if required_name not in globals():
        raise RuntimeError(f"Required object '{required_name}' not found. Run previous cells first.")

metadata = pd.read_csv(RUN_METADATA_CSV, encoding="utf-8-sig")

required_metadata_cols = [
    "source_run_id",
    "source_run_start_date",
    "source_unit_label",
    "source_group_label",
    "source_feature_12",
    "source_feature_13",
    "source_feature_14",
    "source_feature_15",
    SOURCE_TARGET_COL,
]
missing = [col for col in required_metadata_cols if col not in metadata.columns]
if missing:
    raise KeyError(f"Run metadata file is missing expected columns: {missing}")

metadata["run_id"] = metadata["source_run_id"].map(normalize_run_id)
metadata_small = metadata.rename(columns={
    "source_run_start_date": "run_start_date",
    "source_unit_label": "unit_label",
    "source_group_label": "group_label",
    "source_feature_12": "feature_12",
    "source_feature_13": "feature_13",
    "source_feature_14": "feature_14",
    "source_feature_15": "feature_15",
    SOURCE_TARGET_COL: PRIMARY_TARGET_COL,
})[
    [
        "run_id",
        "run_start_date",
        "unit_label",
        "group_label",
        "feature_12",
        "feature_13",
        "feature_14",
        "feature_15",
        PRIMARY_TARGET_COL,
    ]
].copy()

duration_rows = []
for run_id, run_data in enhanced_feature_data.items():
    hours = float(run_data["metadata"]["total_duration_h"]) if np.isfinite(run_data["metadata"]["total_duration_h"]) else np.nan
    duration_rows.append({"run_id": run_id, "total_run_days": (hours / 24.0) if np.isfinite(hours) else np.nan})
duration_df = pd.DataFrame(duration_rows)

feat_df = feat_df.rename(columns={"run_id": "run_id"})
feat_df["run_id"] = feat_df["run_id"].map(normalize_run_id)

feat_df = feat_df.merge(metadata_small, on="run_id", how="left")
feat_df = feat_df.merge(duration_df, on="run_id", how="left")

front_cols = [
    "run_id",
    "feature_1_setpoint",
    "run_start_date",
    "unit_label",
    "group_label",
    "total_run_days",
    "feature_12",
    "feature_13",
    "feature_14",
    "feature_15",
]
for col in front_cols + [PRIMARY_TARGET_COL]:
    if col not in feat_df.columns:
        feat_df[col] = np.nan

feature_cols = [col for col in feat_df.columns if col not in front_cols + [PRIMARY_TARGET_COL]]
feat_df = feat_df[front_cols + feature_cols + [PRIMARY_TARGET_COL]]

FEATURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
wide_csv = FEATURE_OUTPUT_DIR / FEATURE_TABLE_CSV
feat_df.to_csv(wide_csv, index=False)
print(f"Joined run metadata. Saved -> {wide_csv.resolve()}")

full_csv = FEATURE_OUTPUT_DIR / FULL_FEATURE_TABLE_CSV
feat_df.to_csv(full_csv, index=False)
print(f"Full feature table saved -> {full_csv.resolve()}")

exclude_cols = ["run_id", "run_start_date", "unit_label", PRIMARY_TARGET_COL]
X = feat_df.drop(columns=exclude_cols, errors="ignore")
y = feat_df[PRIMARY_TARGET_COL]

X_csv = FEATURE_OUTPUT_DIR / TRAIN_FEATURES_CSV
y_csv = FEATURE_OUTPUT_DIR / TRAIN_TARGET_CSV
X.to_csv(X_csv, index=False)
y.to_csv(y_csv, index=False)

manifest = {
    "feature_table": FEATURE_TABLE_CSV,
    "full_feature_table": FULL_FEATURE_TABLE_CSV,
    "training_features": TRAIN_FEATURES_CSV,
    "training_target": TRAIN_TARGET_CSV,
    "target_column": PRIMARY_TARGET_COL,
    "run_id_column": "run_id",
    "date_column": "run_start_date",
}
with open(FEATURE_OUTPUT_DIR / "pipeline_manifest.json", "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

print(f"Training features saved -> {X_csv.resolve()}")
print(f"Training target saved   -> {y_csv.resolve()}")

feat_df.head()

In [ ]:
# Add optional run-level treatment metrics to the wide and training tables.

for required_name in ("feat_df", "FEATURE_OUTPUT_DIR"):
    if required_name not in globals():
        raise RuntimeError(f"Required object '{required_name}' not found. Run previous blocks first.")

TREATMENT_METADATA_CSV = Path("pipeline_inputs/run_treatment_metrics.csv")
TREATMENT_COLS = [
    "source_run_id",
    "feature_16",
    "feature_17",
]

def coerce_numeric_series(series: pd.Series) -> pd.Series:
    """Coerce text-like numeric values to floats."""
    cleaned = series.astype(str).str.strip().str.replace("[,%]", "", regex=True)
    return pd.to_numeric(cleaned, errors="coerce")

treat = pd.read_csv(TREATMENT_METADATA_CSV, encoding="utf-8-sig")
missing = [col for col in TREATMENT_COLS if col not in treat.columns]
if missing:
    raise KeyError(f"Treatment metrics file is missing expected columns: {missing}")

treat = treat[TREATMENT_COLS].copy()
treat["run_id"] = treat["source_run_id"].map(normalize_run_id)
treat["feature_16"] = coerce_numeric_series(treat["feature_16"])
treat["feature_17"] = coerce_numeric_series(treat["feature_17"])
treat_small = treat[["run_id", "feature_16", "feature_17"]].drop_duplicates("run_id", keep="first")

pre_cols = set(feat_df.columns)
feat_df = feat_df.merge(treat_small, on="run_id", how="left")
new_cols = [col for col in feat_df.columns if col not in pre_cols]

print(f"Added treatment metrics to wide table: {new_cols if new_cols else 'none'}")

FEATURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
wide_csv = FEATURE_OUTPUT_DIR / FEATURE_TABLE_CSV
full_csv = FEATURE_OUTPUT_DIR / FULL_FEATURE_TABLE_CSV
feat_df.to_csv(wide_csv, index=False)
feat_df.to_csv(full_csv, index=False)

exclude_cols = ["run_id", "run_start_date", "unit_label", PRIMARY_TARGET_COL]
X = feat_df.drop(columns=exclude_cols, errors="ignore")
y = feat_df[PRIMARY_TARGET_COL]

X_csv = FEATURE_OUTPUT_DIR / TRAIN_FEATURES_CSV
y_csv = FEATURE_OUTPUT_DIR / TRAIN_TARGET_CSV
X.to_csv(X_csv, index=False)
y.to_csv(y_csv, index=False)
print(f"Training features updated -> {X_csv.resolve()}")
print(f"Training target updated   -> {y_csv.resolve()}")

In [ ]:
# Add optional external run-level metrics.

for required_name in ("feat_df", "FEATURE_OUTPUT_DIR"):
    if required_name not in globals():
        raise RuntimeError(f"Required object '{required_name}' not found. Run previous blocks first.")

EXTERNAL_METRICS_CSV = Path("pipeline_inputs/external_run_metrics.csv")

EXTERNAL_METRIC_A_COLS = ["source_run_id", "source_category", "source_status", "source_metric_a"]
EXTERNAL_METRIC_B_COLS = ["source_run_id", "source_category", "source_metric_b"]

METRIC_A_FILTERS = [
    ("category_group_a", "status_group_a"),
    ("category_group_b", "status_group_b"),
    ("category_group_c", "status_group_c"),
]
METRIC_B_CATEGORY = "category_group_c"

def coerce_external_numeric(series: pd.Series) -> pd.Series:
    """Coerce external metric values to floats."""
    text = series.astype(str).str.strip()
    text = text.replace(
        to_replace=[r"(?i)tntc", r"(?i)too numerous.*", r"(?i)nd", r"^\s*$"],
        value=np.nan,
        regex=True,
    )
    cleaned = text.str.replace(r"[^0-9eE\.\-\+]", "", regex=True)
    return pd.to_numeric(cleaned, errors="coerce")

external = pd.read_csv(EXTERNAL_METRICS_CSV, encoding="utf-8-sig")

missing_a = [col for col in EXTERNAL_METRIC_A_COLS if col not in external.columns]
missing_b = [col for col in EXTERNAL_METRIC_B_COLS if col not in external.columns]
if missing_a:
    raise KeyError(f"External metrics file is missing expected metric-A columns: {missing_a}")
if missing_b:
    raise KeyError(f"External metrics file is missing expected metric-B columns: {missing_b}")

metric_a = external[EXTERNAL_METRIC_A_COLS].copy()
metric_a["run_id"] = metric_a["source_run_id"].map(normalize_run_id)
metric_a["feature_18_val"] = coerce_external_numeric(metric_a["source_metric_a"])

metric_a_mask = False
for category_value, status_value in METRIC_A_FILTERS:
    metric_a_mask = metric_a_mask | (
        (metric_a["source_category"] == category_value) &
        (metric_a["source_status"] == status_value)
    )

feature_18 = (
    metric_a[metric_a_mask]
    .groupby("run_id", as_index=False)["feature_18_val"]
    .mean()
    .rename(columns={"feature_18_val": "feature_18"})
)

pre_cols = set(feat_df.columns)
feat_df = feat_df.merge(feature_18, on="run_id", how="left")
print(f"Added external metric A columns: {[col for col in feat_df.columns if col not in pre_cols] or 'none'}")

metric_b = external[EXTERNAL_METRIC_B_COLS].copy()
metric_b["run_id"] = metric_b["source_run_id"].map(normalize_run_id)
metric_b["feature_19_val"] = coerce_external_numeric(metric_b["source_metric_b"])

feature_19 = (
    metric_b[metric_b["source_category"] == METRIC_B_CATEGORY]
    .groupby("run_id", as_index=False)["feature_19_val"]
    .mean()
    .rename(columns={"feature_19_val": "feature_19"})
)

pre_cols = set(feat_df.columns)
feat_df = feat_df.merge(feature_19, on="run_id", how="left")
print(f"Added external metric B columns: {[col for col in feat_df.columns if col not in pre_cols] or 'none'}")

FEATURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
wide_csv = FEATURE_OUTPUT_DIR / FEATURE_TABLE_CSV
full_csv = FEATURE_OUTPUT_DIR / FULL_FEATURE_TABLE_CSV
feat_df.to_csv(wide_csv, index=False)
feat_df.to_csv(full_csv, index=False)

exclude_cols = ["run_id", "run_start_date", "unit_label", PRIMARY_TARGET_COL]
X = feat_df.drop(columns=exclude_cols, errors="ignore")
y = feat_df[PRIMARY_TARGET_COL]

X_csv = FEATURE_OUTPUT_DIR / TRAIN_FEATURES_CSV
y_csv = FEATURE_OUTPUT_DIR / TRAIN_TARGET_CSV
X.to_csv(X_csv, index=False)
y.to_csv(y_csv, index=False)
print(f"Training features updated -> {X_csv.resolve()}")
print(f"Training target updated   -> {y_csv.resolve()}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Rolling RF: residuals, inner CV RMSE, OOB R² + trend lines  ║
# ╚══════════════════════════════════════════════════════════════╝
import warnings
warnings.filterwarnings("ignore")

# ---------- Config -----------------
# Training window
ROLL_N = 15 # Rolling window size for local training
N_SPLITS = 5  # inner CV splits within the n-run training window

# Driver analysis
HIGH_OOB_R2 = 0.10     # OOB R2 threshold for triggering driver analysis
TOPK_MDI     = 10      # how many features to keep for targeted PI
PI_REPEATS   = 3       # small repeats for speed
PI_HOLD_FRAC = 0.25    # use last % of the training window as a quick holdout

# --- NEW: hygiene filtering (window-specific) ---
MAX_NAN_FRAC   = 0.70  # drop features with >70% missing in the training window
MIN_UNIQUE_NUM = 2     # drop numeric features with <2 unique finite values in the window
# -----------------------------------------------

# ---------- Load wide table (use in-memory feat_df if present) ----------
def _load_wide():
    if "feat_df" in globals():
        return feat_df.copy()
    for p in [
        str(FEATURE_OUTPUT_DIR / FEATURE_TABLE_CSV),
        "feature_table.csv",
    ]:
        try:
            return pd.read_csv(p)
        except Exception:
            pass
    raise FileNotFoundError("Could not find wide features CSV and feat_df is not in memory.")
df = _load_wide()

# ---------- Identify columns ----------
def _find_col(cols, key_substr):
    hits = [c for c in cols if key_substr.lower() in c.lower()]
    return hits[0] if hits else None

run_col    = "run_id"
start_col  = _find_col(df.columns, "run_start_date") or _find_col(df.columns, "Run Start Date")
days_col   = _find_col(df.columns, "total_run_days") or _find_col(df.columns, "Total Run Days")
target_col = PRIMARY_TARGET_COL if PRIMARY_TARGET_COL in df.columns else _find_col(df.columns, "target_metric")
if start_col is None or target_col is None:
    raise RuntimeError(f"Required columns missing. start_col={start_col}, target_col={target_col}")

# ---------- % → numeric coercion ----------
def _to_fraction_series(s: pd.Series) -> pd.Series:
    st = s.astype(str).str.strip()
    has_pct = st.str.contains("%", na=False)
    cleaned = st.str.replace("[,%]", "", regex=True)
    num = pd.to_numeric(cleaned, errors="coerce")
    num = np.where(has_pct, num / 100.0, num)
    return pd.Series(num, index=s.index, dtype="float64")

# Coerce target, and coerce %/MC-looking features if needed
df[target_col] = _to_fraction_series(df[target_col])
percent_name_regex = re.compile(r"%|\bpercent\b|\bmc\b", flags=re.I)
for col in df.columns:
    if col in (run_col, start_col, target_col) or (days_col and col == days_col):
        continue
    if percent_name_regex.search(col) or (df[col].dtype == object and df[col].astype(str).str.contains("%").any()):
        try:
            df[col] = _to_fraction_series(df[col])
        except Exception:
            pass

# ---------- Sort and build feature set ----------
df[start_col] = pd.to_datetime(df[start_col], errors="coerce")
df = df[df[target_col].notna()].copy()
df = df.sort_values([start_col, run_col]).reset_index(drop=True)

exclude = {run_col, start_col, target_col}
if days_col: exclude.add(days_col)
feature_cols = [c for c in df.columns if c not in exclude]

# --- NEW: window-specific hygiene filtering helpers ---
def _hygiene_keep_cols(X_trwin: pd.DataFrame, candidate_cols: list[str]) -> list[str]:
    """Drop high-missing and near-constant numeric features within the training window."""
    keep = []
    for c in candidate_cols:
        s = X_trwin[c]

        # 1) Missingness filter (applies to both numeric + categorical)
        nan_frac = float(s.isna().mean())
        if np.isfinite(nan_frac) and nan_frac > MAX_NAN_FRAC:
            continue

        # 2) Near-constant numeric filter (within window)
        if pd.api.types.is_numeric_dtype(s):
            finite = pd.to_numeric(s, errors="coerce")
            finite = finite[np.isfinite(finite)]
            if finite.size == 0:
                continue
            if finite.nunique(dropna=True) < MIN_UNIQUE_NUM:
                continue

        keep.append(c)

    return keep
# -----------------------------------------------------

# --- Helpers for feature importance ---

def _expanded_feature_names(prep):
    """
    Get transformed feature names aligned to rf.feature_importances_.
    Works for ColumnTransformer with ('num', ...) and ('cat', .../OneHotEncoder).
    """
    names = []
    for name, trans, cols in prep.transformers_:
        if name == "remainder" or trans == "drop":
            continue
        step = trans
        if hasattr(trans, "named_steps"):
            # grab onehot step if present
            step = trans.named_steps.get("onehot", trans)
        if hasattr(step, "get_feature_names_out"):
            out = step.get_feature_names_out(cols)
        else:
            out = np.array(cols, dtype=str)
        names.extend(list(out))
    return np.array(names, dtype=str)

def _aggregate_to_raw(importances, transformed_names, num_cols_parent, cat_cols_parent):
    """
    Aggregate transformed importances back to raw feature names.
    - Numeric features: 1:1 (leave the name unchanged).
    - Categorical (one-hot): map 'col_<level>' or 'col=<level>' back to 'col'.
    """
    parents = []
    cat_set = set(cat_cols_parent)

    for n in transformed_names:
        parent = None
        # map OHE outputs back to their categorical parent by prefix
        for c in cat_set:
            if n.startswith(f"{c}_") or n.startswith(f"{c}="):
                parent = c
                break
        if parent is None:
            # treat as numeric passthrough (keep full name unchanged)
            parent = n
        parents.append(parent)

    s = pd.Series(importances, index=parents).groupby(level=0).sum().sort_values(ascending=False)
    return s

def _thin_pipeline_from_parent_cols(preprocess, all_num_cols, all_cat_cols, keep_raw_cols):
    """Build a tiny ColumnTransformer that keeps only K raw columns (fast to fit)."""
    keep_num = [c for c in all_num_cols if c in keep_raw_cols]
    keep_cat = [c for c in all_cat_cols if c in keep_raw_cols]

    num_small = Pipeline([("imputer", preprocess.named_transformers_["num"].named_steps["imputer"])])
    cat_small = Pipeline([
        ("imputer", preprocess.named_transformers_["cat"].named_steps["imputer"]),
        ("onehot",  preprocess.named_transformers_["cat"].named_steps["onehot"]),
    ])
    small_prep = ColumnTransformer([
        ("num", num_small, keep_num),
        ("cat", cat_small, keep_cat),
    ], remainder="drop")

    small_rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    return Pipeline([("prep", small_prep), ("rf", small_rf)]), keep_num, keep_cat

driver_log = []   # collects snapshots when OOB R² is HIGH

# ---------- Rolling walk-forward with inner TimeSeries CV ----------

runs  = df[run_col].tolist()
dates = df[start_col].tolist()
X_all = df[feature_cols]
y_all = df[target_col]

if len(df) <= ROLL_N:
    raise RuntimeError(f"Need > {ROLL_N} runs; only found {len(df)} usable rows.")

records = []
for i in range(ROLL_N, len(df)):
    train_idx = list(range(i - ROLL_N, i))  # prior n run window
    test_idx  = [i]                         # next run

    X_trwin, y_trwin = X_all.iloc[train_idx], y_all.iloc[train_idx]
    X_test,  y_test  = X_all.iloc[test_idx],  y_all.iloc[test_idx]

    # --- hygiene-filter feature list for this window ---
    win_cols = _hygiene_keep_cols(X_trwin, feature_cols)

    # OPTIONAL (1-line) hygiene log:
    #print(f"🧹 hygiene window {runs[i]}: kept {len(win_cols)}/{len(feature_cols)} features (dropped {len(feature_cols)-len(win_cols)})")

    # rebuild preprocess for this window only (minimal changes; uses same logic)
    num_cols = [c for c in win_cols if pd.api.types.is_numeric_dtype(df[c])]
    cat_cols = [c for c in win_cols if c not in num_cols]

    numeric_transformer = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    preprocess = ColumnTransformer([
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ], remainder="drop")

    base_model = Pipeline([
        ("prep", preprocess),
        ("rf", RandomForestRegressor(
            n_estimators=500, random_state=42, n_jobs=-1,
            oob_score=True, bootstrap=True
        ))
    ])

    # ----- Inner time-series CV on the training window (RMSE only) -----
    tss = TimeSeriesSplit(n_splits=min(N_SPLITS, max(2, len(train_idx) - 1)))
    cv_rmses = []
    for tr_idx_rel, val_idx_rel in tss.split(X_trwin):
        X_tr, X_val = X_trwin.iloc[tr_idx_rel][win_cols], X_trwin.iloc[val_idx_rel][win_cols]
        y_tr, y_val = y_trwin.iloc[tr_idx_rel], y_trwin.iloc[val_idx_rel]

        model_cv = Pipeline([
            ("prep", preprocess),
            ("rf", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1))
        ])
        model_cv.fit(X_tr, y_tr)
        y_val_pred = model_cv.predict(X_val)
        cv_rmses.append(mean_squared_error(y_val, y_val_pred, squared=False))

    cv_rmse = float(np.mean(cv_rmses)) if cv_rmses else np.nan

    # ----- Fit full window (with OOB) and predict the held-out run -----
    model = base_model
    model.fit(X_trwin[win_cols], y_trwin)
    y_pred = float(model.predict(X_test[win_cols])[0])
    y_true = float(y_test.iloc[0])
    resid  = y_true - y_pred

    # OOB R² on training window
    try:
        oob_r2 = float(model.named_steps["rf"].oob_score_)
    except Exception:
        oob_r2 = np.nan

    records.append({
        "run_id": runs[i],
        "run_start_date": dates[i],
        "residual": resid,
        "cv_rmse": cv_rmse,
        "oob_r2": oob_r2
    })

    # ---------- GATED DRIVERS (run only when OOB R² is high) ----------
    if np.isfinite(oob_r2) and oob_r2 >= HIGH_OOB_R2:
        # 1) Fast MDI (built-in) → aggregate to raw feature names
        prep = model.named_steps["prep"]
        rf_   = model.named_steps["rf"]
        tf_names = _expanded_feature_names(prep)

        # window-specific lists for robust mapping
        num_cols_win = [c for c in num_cols if c in X_trwin.columns]
        cat_cols_win = [c for c in cat_cols if c in X_trwin.columns]

        mdi_raw  = _aggregate_to_raw(rf_.feature_importances_, tf_names, num_cols_win, cat_cols_win)

        # Keep a compact top-K list
        topk = mdi_raw.head(TOPK_MDI)
        topk_list = list(zip(topk.index.tolist(), topk.values.tolist()))

        # 2) Tiny, targeted permutation importance (re-fit on top-K only)
        #    Use last PI_HOLD_FRAC of the training window as a quick holdout
        hold_start = int(round((1.0 - PI_HOLD_FRAC) * len(X_trwin)))
        X_hold = X_trwin.iloc[hold_start:][win_cols]
        y_hold = y_trwin.iloc[hold_start:]

        keep_cols = [c for (c, _) in topk_list if c in win_cols]
        if not keep_cols:
            # Log MDI snapshot and move on; skip PI for this iteration
            driver_log.append({
                "run_id": runs[i],
                "run_start_date": dates[i],
                "oob_r2": oob_r2,
                "topk_mdi_raw": topk_list,
                "pi_r2_drop_topk": []
            })
            continue

        thin_pipe, keep_num, keep_cat = _thin_pipeline_from_parent_cols(
            prep, num_cols_win, cat_cols_win, keep_cols
        )

        # Fit small model and compute PI on the small holdout
        thin_pipe.fit(X_trwin[keep_cols], y_trwin)
        pi = permutation_importance(
            thin_pipe, X_hold[keep_cols], y_hold,
            n_repeats=PI_REPEATS, random_state=42, n_jobs=-1, scoring="r2"
        )
        pi_mean = pd.Series(pi.importances_mean, index=keep_cols).sort_values(ascending=False)

        driver_log.append({
            "run_id": runs[i],
            "run_start_date": dates[i],
            "oob_r2": oob_r2,
            "topk_mdi_raw": topk_list,                       # [(feature, mdi), ...]
            "pi_r2_drop_topk": list(pi_mean.items()),        # [(feature, mean ΔR²), ...]
        })

metrics_df = pd.DataFrame(records).sort_values("run_start_date").reset_index(drop=True)

# ---------- Driver summary / report (ADDED) ----------
def _fmt_top(items, k=5):
    try:
        return ", ".join(f"{f}({v:.3f})" for f, v in items[:k])
    except Exception:
        return ""

if len(driver_log):
    print(f"🔎 Driver snapshots captured for {len(driver_log)} runs (OOB R² ≥ {HIGH_OOB_R2}).")
    drivers_report = pd.DataFrame([{
        "run_id": d["run_id"],
        "run_start_date": d["run_start_date"],
        "oob_r2": d["oob_r2"],
        "topk_mdi_raw": _fmt_top(d["topk_mdi_raw"]),
        "pi_r2_drop_topk": _fmt_top(d["pi_r2_drop_topk"])
    } for d in driver_log]).sort_values("run_start_date").reset_index(drop=True)

    # Save and show a compact tail
    out_dir = Path("pipeline_outputs/features")
    out_dir.mkdir(parents=True, exist_ok=True)
    drivers_csv = out_dir / "model_driver_snapshots.csv"
    drivers_report.to_csv(drivers_csv, index=False)
    print(f"📄 Driver snapshots saved → {drivers_csv.resolve()}")
    print(drivers_report.tail(10).to_string(index=False))
else:
    print(f"ℹ️ No driver snapshots captured (no OOB R² ≥ {HIGH_OOB_R2}).")

# ---------- Helpers for trend overlays ----------
def _plot_with_trend(ax, x_num, y, label):
    """Plot series + LOWESS smooth + global linear trend."""
    ax.plot(x_num, y, marker="o", lw=1.2, label=label)
    valid = np.isfinite(y)
    if valid.sum() >= 3:
        # LOWESS smooth (frac auto-scales with length; clamp to [0.2, 0.6])
        frac = np.clip(10 / len(y), 0.2, 0.6)
        lo = lowess(y[valid], x_num[valid], frac=frac, return_sorted=True)
        ax.plot(lo[:, 0], lo[:, 1], lw=2.0, alpha=0.9, linestyle="-", label=f"{label} (LOWESS)")
        # Linear trend
        m, b = np.polyfit(x_num[valid], y[valid], 1)
        ax.plot([x_num[valid][0], x_num[valid][-1]],
                [m * x_num[valid][0] + b, m * x_num[valid][-1] + b],
                lw=1.5, linestyle=":", label=f"{label} (linear trend)")

# Use numeric x for consistent overlays; label with Run strings
x_num = np.arange(len(metrics_df))
x_labels = metrics_df["run_id"].tolist()

# ---------- Plots ----------
# 1) OOS residuals + trends
plt.figure(figsize=(14, 5))
plt.axhline(0, lw=1, color="k", alpha=0.6)
ax = plt.gca()
_plot_with_trend(ax, x_num, metrics_df["residual"].to_numpy(), "OOS residual (obs−pred)")
plt.ylabel("Residual")
plt.title(f"Rolling RF: Out-of-sample residuals (window={ROLL_N})")
plt.xticks(ticks=x_num, labels=x_labels, rotation=90)
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

# 2) Inner CV RMSE + trends
#plt.figure(figsize=(14, 5))
#ax = plt.gca()
#_plot_with_trend(ax, x_num, metrics_df["cv_rmse"].to_numpy(), "Inner CV RMSE")
#plt.ylabel("RMSE")
#plt.title("Inner time-series CV RMSE on training window")
#plt.xticks(ticks=x_num, labels=x_labels, rotation=90)
#plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

# 3) OOB R² + trends
plt.figure(figsize=(14, 5))
ax = plt.gca()
_plot_with_trend(ax, x_num, metrics_df["oob_r2"].to_numpy(), "OOB R²")

# add horizontal line at R² = 0
ax.axhline(0, lw=1, color="k", alpha=0.6, linestyle="-")

plt.ylabel("OOB R²")
plt.title("Random Forest OOB R² on training window")
plt.xticks(ticks=x_num, labels=x_labels, rotation=90)
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()